In [2]:
import random
import time
# %pip install "gymnasium[classic-control]"
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as opt

In [13]:
# settings
env_id = "CartPole-v1"

# number of environments to run in parallel
num_envs = 1

# total number of timesteps for training over multiple episodes
total_timesteps = 100_000

# adam learning rate
learning_rate = 1e-3

# discount factor
gamma = 0.99

# epsilon value for epsilon-greedy
eps = 0.1

# seed for reproducibility
seed = None

video_path = "qlearning_cartpole_videos"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "cpu"
)

In [5]:
def make_env(env_id, capture_video, seed=None):
    if capture_video:
        env = gym.make(env_id, render_mode="rgb_array")
        env = gym.wrappers.RecordVideo(env, video_path, episode_trigger=lambda episode: True)
    else:
        env = gym.make(env_id)

    env = gym.wrappers.RecordEpisodeStatistics(env)

    if seed is not None:
        env.action_space.seed(seed)

    return env

In [15]:
# NN
class QNetwork(nn.Module):

    def __init__(self, envs, n_hidden=128):
        super().__init__()

        self.network = nn.Sequential(
                        nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden),
                        nn.ReLU(),
                        nn.Linear(n_hidden, envs.single_action_space.n)
        )

    def forward(self, state_x):
        return self.network(state_x)

def np2torch(a):
    return torch.Tensor(a).float().to(device)

In [ ]:
# make environaments
envs = gym.vector.SyncVectorEnv(
    [
        lambda i=i: make_env(
            env_id,
            False,
            seed=None if seed is None else seed + i,
        )
        for i in range(num_envs)
    ]
)

In [ ]:
episode_returns = []
losses = []
start_time = time.time()
obs, _ = envs.reset(seed=seed)
auto_reset = np.zeros(num_envs, dtype = bool)